# Notebook — Agent LLM pour entités nommées et réinjection TEI inline

Ce notebook propose un workflow semi-automatique :

1. extraction des lignes du fichier XML-TEI ;
2. proposition d'entités nommées par un agent LLM local avec Agno/Ollama ;
3. correction humaine facultative dans une interface Jupyter ;
4. création d'un JSON pivot avec offsets ;
5. réinjection des balises TEI directement dans le texte.

Les balises produites sont intégrées inline dans le TEI, par exemple :

```xml
<line>Mon Pere alla à <placeName>Reconvillier</placeName> avec <persName>Marion</persName>.</line>
```

In [1]:
# Cellule 1 — Imports

from pathlib import Path
from lxml import etree

from agno.agent import Agent
from agno.models.ollama import Ollama

from pydantic import BaseModel, Field
from typing import List, Literal

import json
import re

import ipywidgets as widgets
from IPython.display import display

In [2]:
# Cellule 2 — Configuration générale

DOSSIER_DATA = Path("../data")

FICHIER_TEI = DOSSIER_DATA / "Frêne_volume_1.xml"

FICHIER_JSON_LIGNES = DOSSIER_DATA / "lignes_tei_entites.json"
FICHIER_JSON_PROPOSITIONS = DOSSIER_DATA / "propositions_entites_llm.json"
FICHIER_JSON_CORRIGE = DOSSIER_DATA / "propositions_entites_corrigees.json"
FICHIER_JSON_PIVOT = DOSSIER_DATA / "entites_pivot_offsets.json"

FICHIER_SORTIE = DOSSIER_DATA / "Frêne_volume_1_entites_inline.xml"

MODELE_OLLAMA = "qwen3:8b"

NS_TEI = "http://www.tei-c.org/ns/1.0"
XML_NS = "http://www.w3.org/XML/1998/namespace"

NS = {
    "tei": NS_TEI,
    "xml": XML_NS,
}

XML_ID = f"{{{XML_NS}}}id"

TYPES_TEI_AUTORISES = {
    "persName",
    "placeName",
    "orgName",
    "date",
    "roleName",
}

print("Configuration chargée.")

Configuration chargée.


In [3]:
# Cellule 3 — Chargement du fichier XML-TEI

def charger_tei(fichier: Path) -> etree._ElementTree:
    """Charge un fichier XML-TEI en conservant les espaces autant que possible."""

    parser = etree.XMLParser(
        remove_blank_text=False,
        recover=True
    )

    return etree.parse(
        str(fichier),
        parser
    )


arbre = charger_tei(FICHIER_TEI)
racine = arbre.getroot()

print(f"Fichier TEI chargé : {FICHIER_TEI}")

Fichier TEI chargé : ..\data\Frêne_volume_1.xml


In [4]:
# Cellule 4 — Extraction des lignes TEI

def selectionner_elements_lignes(arbre: etree._ElementTree) -> list:
    """Sélectionne les lignes TEI.

    La fonction cherche d'abord les éléments <line>, fréquents dans <sourceDoc>.
    Si aucun élément <line> n'est trouvé, elle cherche les éléments <l>.
    """

    lignes = arbre.xpath(
        "//tei:line",
        namespaces=NS
    )

    if not lignes:
        lignes = arbre.xpath(
            "//tei:l",
            namespaces=NS
        )

    return lignes


def extraire_lignes_tei(arbre: etree._ElementTree) -> list[dict]:
    """Extrait les lignes du TEI avec leur xml:id et leur texte."""

    lignes = selectionner_elements_lignes(arbre)
    resultat = []

    for i, ligne in enumerate(lignes, start=1):
        xml_id = ligne.get(XML_ID)

        if xml_id is None:
            xml_id = f"ligne_{i:06d}"
            ligne.set(XML_ID, xml_id)

        texte = "".join(ligne.itertext()).strip()

        if texte:
            resultat.append({
                "xml_id": xml_id,
                "numero": i,
                "texte": texte
            })

    return resultat


lignes_json = extraire_lignes_tei(arbre)

with open(FICHIER_JSON_LIGNES, "w", encoding="utf-8") as f:
    json.dump(
        lignes_json,
        f,
        ensure_ascii=False,
        indent=2
    )

print(f"{len(lignes_json)} lignes extraites.")
print(f"JSON des lignes créé : {FICHIER_JSON_LIGNES}")

175 lignes extraites.
JSON des lignes créé : ..\data\lignes_tei_entites.json


In [5]:
# Cellule 5 — Schéma Pydantic pour les propositions d'entités

class PropositionEntite(BaseModel):
    """Entité nommée proposée par le LLM."""

    surface: str = Field(
        ...,
        description="Forme exacte présente dans le texte."
    )

    type: Literal[
        "persName",
        "placeName",
        "orgName",
        "date",
        "roleName"
    ]

    confidence: Literal[
        "certain",
        "probable",
        "incertain"
    ] = "probable"

    commentaire: str = ""


class ListeEntites(BaseModel):
    """Liste d'entités nommées proposées par le LLM."""

    entites: List[PropositionEntite]

In [6]:
# Cellule 6 — Création de l'agent LLM

agent_entites = Agent(
    model=Ollama(
        id=MODELE_OLLAMA
    ),

    output_schema=ListeEntites,

    instructions=[
        "Tu es un assistant spécialisé dans l'annotation TEI de textes manuscrits anciens en français.",
        "Tu travailles sur un journal manuscrit jurassien du XVIIIe siècle.",
        "Ta tâche est de repérer les entités nommées explicitement présentes dans le texte.",
        "Ne corrige pas le texte.",
        "Ne modernise pas l'orthographe.",
        "Ne reformule pas.",
        "La valeur de surface doit être copiée exactement depuis le texte.",
        "Ne donne pas d'offsets.",
        "Utilise uniquement les types TEI suivants : persName, placeName, orgName, date, roleName.",
        "persName désigne une personne, un prénom, un nom ou une désignation familiale individualisée.",
        "placeName désigne un lieu, village, ville, région, pays ou bâtiment localisé.",
        "orgName désigne une institution, une autorité constituée ou une organisation.",
        "date désigne une date explicite, une année, un mois ou une fête datante.",
        "roleName désigne une fonction ou un titre social associé à une personne.",
        "Si aucune entité n'est présente, retourne une liste vide.",
        "Retourne uniquement la structure JSON attendue."
    ],
)

print("Agent LLM créé.")

Agent LLM créé.


In [7]:
# Cellule 7 — Fonction de proposition d'entités pour une ligne

def proposer_entites_ligne(ligne: dict) -> dict:
    """Demande au LLM de proposer les entités nommées présentes dans une ligne."""

    prompt = f"""
Analyse cette ligne issue d'un fichier XML-TEI.

xml_id : {ligne["xml_id"]}

Texte :
{ligne["texte"]}

Repère uniquement les entités nommées explicitement présentes dans le texte.
"""

    try:
        reponse = agent_entites.run(prompt)
        contenu = reponse.content

        if isinstance(contenu, ListeEntites):
            entites = [
                entite.model_dump()
                for entite in contenu.entites
            ]

        elif isinstance(contenu, dict):
            entites = contenu.get("entites", [])

        else:
            entites = []

        erreur = None

    except Exception as exception:
        entites = []
        erreur = str(exception)

    return {
        "xml_id": ligne["xml_id"],
        "numero": ligne.get("numero"),
        "texte": ligne["texte"],
        "entites_proposees": entites,
        "erreur": erreur
    }

In [8]:
# Cellule 8 — Génération des propositions LLM

# Pour un premier test, il est conseillé de limiter le nombre de lignes.
# Mets LIMITE_LIGNES = None pour traiter tout le fichier.

LIMITE_LIGNES = 10

if LIMITE_LIGNES is None:
    lignes_a_traiter = lignes_json
else:
    lignes_a_traiter = lignes_json[:LIMITE_LIGNES]

propositions = []

for i, ligne in enumerate(lignes_a_traiter, start=1):
    resultat = proposer_entites_ligne(ligne)
    propositions.append(resultat)

    if i % 10 == 0:
        print(f"{i} lignes traitées...")

with open(FICHIER_JSON_PROPOSITIONS, "w", encoding="utf-8") as f:
    json.dump(
        propositions,
        f,
        ensure_ascii=False,
        indent=2
    )

print(f"{len(propositions)} lignes traitées.")
print(f"JSON des propositions créé : {FICHIER_JSON_PROPOSITIONS}")

10 lignes traitées...
10 lignes traitées.
JSON des propositions créé : ..\data\propositions_entites_llm.json


In [9]:
# Cellule 9 — Interface de correction humaine interactive

with open(FICHIER_JSON_PROPOSITIONS, "r", encoding="utf-8") as f:
    propositions = json.load(f)

# On ne corrige dans l'interface que les lignes contenant des entités proposées.
# Les lignes sans entité restent conservées dans le JSON final.
lignes_a_corriger = [
    ligne for ligne in propositions
    if ligne.get("entites_proposees")
]

index_courant = 0

zone_texte = widgets.HTML()

zone_json = widgets.Textarea(
    layout=widgets.Layout(
        width="100%",
        height="320px"
    )
)

bouton_precedent = widgets.Button(
    description="← Précédent"
)

bouton_suivant = widgets.Button(
    description="Suivant →"
)

bouton_enregistrer_ligne = widgets.Button(
    description="Enregistrer cette ligne",
    button_style="info"
)

bouton_enregistrer_tout = widgets.Button(
    description="Enregistrer le JSON corrigé",
    button_style="success"
)

message = widgets.Output()


def afficher_ligne():
    """Affiche la ligne courante et ses entités proposées."""

    if not lignes_a_corriger:
        zone_texte.value = "<b>Aucune entité proposée à corriger.</b>"
        zone_json.value = "[]"
        return

    ligne = lignes_a_corriger[index_courant]

    zone_texte.value = f"""
    <b>Ligne {index_courant + 1} / {len(lignes_a_corriger)}</b><br>
    <b>xml_id :</b> {ligne["xml_id"]}<br>
    <b>numéro :</b> {ligne.get("numero")}<br><br>
    <b>Texte :</b><br>
    <div style="border:1px solid #cccccc; padding:10px; background-color:#fafafa;">
        {ligne["texte"]}
    </div>
    """

    zone_json.value = json.dumps(
        ligne["entites_proposees"],
        ensure_ascii=False,
        indent=2
    )


def enregistrer_ligne(_=None):
    """Enregistre les corrections de la ligne courante en mémoire."""

    if not lignes_a_corriger:
        return

    try:
        nouvelles_entites = json.loads(zone_json.value)

        if not isinstance(nouvelles_entites, list):
            raise ValueError("Le contenu doit être une liste JSON.")

        lignes_a_corriger[index_courant]["entites_proposees"] = nouvelles_entites

        with message:
            message.clear_output()
            print("Ligne enregistrée en mémoire.")

    except Exception as erreur:
        with message:
            message.clear_output()
            print(f"Erreur JSON : {erreur}")


def precedent(_):
    """Passe à la ligne précédente."""

    global index_courant

    enregistrer_ligne()

    if index_courant > 0:
        index_courant -= 1

    afficher_ligne()


def suivant(_):
    """Passe à la ligne suivante."""

    global index_courant

    enregistrer_ligne()

    if index_courant < len(lignes_a_corriger) - 1:
        index_courant += 1

    afficher_ligne()


def enregistrer_tout(_):
    """Enregistre le JSON corrigé sur disque."""

    enregistrer_ligne()

    index_corrections = {
        ligne["xml_id"]: ligne["entites_proposees"]
        for ligne in lignes_a_corriger
    }

    propositions_corrigees = []

    for ligne in propositions:
        if ligne["xml_id"] in index_corrections:
            ligne["entites_proposees"] = index_corrections[ligne["xml_id"]]
            ligne["correction_verifiee"] = True
        else:
            ligne["correction_verifiee"] = False

        propositions_corrigees.append(ligne)

    with open(FICHIER_JSON_CORRIGE, "w", encoding="utf-8") as f:
        json.dump(
            propositions_corrigees,
            f,
            ensure_ascii=False,
            indent=2
        )

    with message:
        message.clear_output()
        print(f"JSON corrigé enregistré : {FICHIER_JSON_CORRIGE}")


bouton_precedent.on_click(precedent)
bouton_suivant.on_click(suivant)
bouton_enregistrer_ligne.on_click(enregistrer_ligne)
bouton_enregistrer_tout.on_click(enregistrer_tout)

afficher_ligne()

display(zone_texte)
display(zone_json)
display(
    widgets.HBox([
        bouton_precedent,
        bouton_suivant,
        bouton_enregistrer_ligne,
        bouton_enregistrer_tout
    ])
)
display(message)

HTML(value='\n    <b>Ligne 1 / 7</b><br>\n    <b>xml_id :</b> f8-eSc_textblock_2873aa2a-eSc_line_f93609b4-line…

Textarea(value='[\n  {\n    "surface": "Chevres",\n    "type": "persName",\n    "confidence": "probable",\n   …

Output()

In [10]:
# Cellule 10 — Validation des entités corrigées

def nettoyer_entite(entite: dict) -> dict | None:
    """Nettoie et valide une entité proposée ou corrigée."""

    surface = entite.get("surface")
    type_tei = entite.get("type")

    if not surface or not isinstance(surface, str):
        return None

    if not type_tei or type_tei not in TYPES_TEI_AUTORISES:
        return None

    return {
        "surface": surface,
        "type": type_tei,
        "confidence": entite.get("confidence", "incertain"),
        "commentaire": entite.get("commentaire", "")
    }


def trouver_occurrences(texte: str, surface: str) -> list[dict]:
    """Trouve toutes les occurrences exactes d'une surface dans un texte."""

    return [
        {
            "start": match.start(),
            "end": match.end()
        }
        for match in re.finditer(
            re.escape(surface),
            texte
        )
    ]

In [11]:
# Cellule 11 — Création du JSON pivot avec offsets

def creer_pivot_offsets(donnees_corrigees: list[dict]) -> list[dict]:
    """Crée un JSON pivot avec offsets pour chaque occurrence d'entité."""

    pivot = []
    compteur = 1

    for ligne in donnees_corrigees:
        texte = ligne["texte"]
        entites_finales = []

        for entite_brute in ligne.get("entites_proposees", []):
            entite = nettoyer_entite(entite_brute)

            if entite is None:
                continue

            occurrences = trouver_occurrences(
                texte,
                entite["surface"]
            )

            if not occurrences:
                entites_finales.append({
                    **entite,
                    "status": "non_retrouvee"
                })
                continue

            for occurrence in occurrences:
                entites_finales.append({
                    "entity_id": f"ent_{compteur:06d}",
                    **entite,
                    "start": occurrence["start"],
                    "end": occurrence["end"],
                    "status": "validee"
                })

                compteur += 1

        entites_finales = sorted(
            entites_finales,
            key=lambda e: e.get("start", 10**9)
        )

        pivot.append({
            "xml_id": ligne["xml_id"],
            "numero": ligne.get("numero"),
            "texte": texte,
            "entites": entites_finales
        })

    return pivot


with open(FICHIER_JSON_CORRIGE, "r", encoding="utf-8") as f:
    donnees_corrigees = json.load(f)

pivot = creer_pivot_offsets(donnees_corrigees)

with open(FICHIER_JSON_PIVOT, "w", encoding="utf-8") as f:
    json.dump(
        pivot,
        f,
        ensure_ascii=False,
        indent=2
    )

print(f"JSON pivot créé : {FICHIER_JSON_PIVOT}")

JSON pivot créé : ..\data\entites_pivot_offsets.json


In [12]:
# Cellule 12 — Fonctions de réinjection inline dans le TEI

def filtrer_entites_sans_chevauchement(entites: list[dict]) -> list[dict]:
    """Conserve les entités valides sans chevauchement.

    Si deux entités se chevauchent, la première retenue est la plus longue
    parmi celles qui commencent à la même position.
    """

    valides = [
        entite for entite in entites
        if entite.get("status") == "validee"
        and isinstance(entite.get("start"), int)
        and isinstance(entite.get("end"), int)
    ]

    valides = sorted(
        valides,
        key=lambda e: (
            e["start"],
            -(e["end"] - e["start"])
        )
    )

    resultat = []
    derniere_fin = -1

    for entite in valides:
        if entite["start"] >= derniere_fin:
            resultat.append(entite)
            derniere_fin = entite["end"]

    return resultat


def creer_balise_entite(entite: dict, texte: str) -> etree._Element:
    """Crée une balise TEI inline autour d'une entité."""

    element = etree.Element(
        f"{{{NS_TEI}}}{entite['type']}"
    )

    element.set(
        XML_ID,
        entite["entity_id"]
    )

    if entite.get("confidence"):
        element.set(
            "cert",
            entite["confidence"]
        )

    element.text = texte[
        entite["start"]:
        entite["end"]
    ]

    return element


def fragmenter_texte_inline(texte: str, entites: list[dict]) -> list:
    """Découpe une ligne en fragments texte et balises TEI."""

    fragments = []
    position = 0

    for entite in entites:
        start = entite["start"]
        end = entite["end"]

        if start > position:
            fragments.append(
                texte[position:start]
            )

        fragments.append(
            creer_balise_entite(
                entite,
                texte
            )
        )

        position = end

    if position < len(texte):
        fragments.append(
            texte[position:]
        )

    return fragments


def remplacer_contenu_inline(element: etree._Element, fragments: list) -> None:
    """Remplace le contenu d'un élément TEI par des fragments inline."""

    for enfant in list(element):
        element.remove(enfant)

    element.text = None
    dernier_element = None

    for fragment in fragments:
        if isinstance(fragment, str):
            if dernier_element is None:
                element.text = (
                    element.text or ""
                ) + fragment
            else:
                dernier_element.tail = (
                    dernier_element.tail or ""
                ) + fragment
        else:
            element.append(fragment)
            dernier_element = fragment

In [13]:
# Cellule 13 — Réinjection des entités directement dans les lignes TEI

def indexer_lignes_tei(arbre: etree._ElementTree) -> dict:
    """Indexe les éléments de ligne TEI par xml:id."""

    lignes = selectionner_elements_lignes(arbre)

    return {
        ligne.get(XML_ID): ligne
        for ligne in lignes
        if ligne.get(XML_ID)
    }


def reinjecter_entites_inline(
    arbre: etree._ElementTree,
    fichier_pivot: Path
) -> int:
    """Réinjecte les entités du JSON pivot directement dans le texte TEI."""

    with open(fichier_pivot, "r", encoding="utf-8") as f:
        pivot = json.load(f)

    index_lignes = indexer_lignes_tei(arbre)

    compteur_entites = 0

    for entree in pivot:
        xml_id = entree["xml_id"]
        ligne = index_lignes.get(xml_id)

        if ligne is None:
            continue

        texte = entree["texte"]

        entites = filtrer_entites_sans_chevauchement(
            entree.get("entites", [])
        )

        if not entites:
            continue

        fragments = fragmenter_texte_inline(
            texte,
            entites
        )

        remplacer_contenu_inline(
            ligne,
            fragments
        )

        compteur_entites += len(entites)

    return compteur_entites

In [14]:
# Cellule 14 — Export du TEI enrichi avec balises inline

arbre = charger_tei(FICHIER_TEI)

nombre_entites = reinjecter_entites_inline(
    arbre,
    FICHIER_JSON_PIVOT
)

arbre.write(
    str(FICHIER_SORTIE),
    encoding="UTF-8",
    xml_declaration=True,
    pretty_print=True
)

print(f"{nombre_entites} entités réinjectées directement dans le texte TEI.")
print(f"Fichier TEI enrichi créé : {FICHIER_SORTIE}")

10 entités réinjectées directement dans le texte TEI.
Fichier TEI enrichi créé : ..\data\Frêne_volume_1_entites_inline.xml


## Remarque importante

La réinjection inline remplace le contenu textuel des éléments `<line>` ou `<l>` par un contenu XML mixte.  
Il est donc conseillé de toujours conserver une copie du TEI original avant d'exécuter la cellule d'export.

Les fichiers intermédiaires permettent de documenter la chaîne de traitement :

- `lignes_tei_entites.json`
- `propositions_entites_llm.json`
- `propositions_entites_corrigees.json`
- `entites_pivot_offsets.json`
- `Frêne_volume_1_entites_inline.xml`